# Convolution using only einops, einsum

In [1]:
#! pip install einops einsum

In [78]:
import torch
from torch import randn
#from einsum import einsum
from torch import einsum
from einops import rearrange, repeat, reduce
from torch.nn import Unfold, Module

import itertools
import math

In [84]:
torch.manual_seed(42)
x = torch.randn(2, 3 ,4,4) # BxCxHxW
#

In [67]:
torch.manual_seed(42)
_kernel = torch.randn(10,3,3)
_kernel #fxkxk

tensor([[[ 1.9269,  1.4873,  0.9007],
         [-2.1055,  0.6784, -1.2345],
         [-0.0431, -1.6047, -0.7521]],

        [[ 1.6487, -0.3925, -1.4036],
         [-0.7279, -0.5594, -0.7688],
         [ 0.7624,  1.6423, -0.1596]],

        [[-0.4974,  0.4396, -0.7581],
         [ 1.0783,  0.8008,  1.6806],
         [ 1.2791,  1.2964,  0.6105]],

        [[ 1.3347, -0.2316,  0.0418],
         [-0.2516,  0.8599, -1.3847],
         [-0.8712, -0.2234,  1.7174]],

        [[ 0.3189, -0.4245,  0.3057],
         [-0.7746, -1.5576,  0.9956],
         [-0.8798, -0.6011, -1.2742]],

        [[ 2.1228, -1.2347, -0.4879],
         [-0.9138, -0.6581,  0.0780],
         [ 0.5258, -0.4880,  1.1914]],

        [[-0.8140, -0.7360, -1.4032],
         [ 0.0360, -0.0635,  0.6756],
         [-0.0978,  1.8446, -1.1845]],

        [[ 1.3835,  1.4451,  0.8564],
         [ 2.2181,  0.5232,  0.3466],
         [-0.1973, -1.0546,  1.2780]],

        [[-0.1722,  0.5238, -0.6021],
         [ 0.9604,  0.4048, -1.354

In [153]:
class EinConv2d(Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, dilation=1,padding=0, seed=42):
        super(EinConv2d, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels

        self.kernel_size = kernel_size if type(kernel_size) == tuple else (kernel_size, kernel_size)
        self.stride = stride if type(stride) == tuple else (stride, stride)
        self.dilation = dilation if type(dilation) == tuple else (dilation, dilation)
        self.padding = padding if type(padding) == tuple else (padding, padding)

        self.kernel_seed = seed
        self._init_kernel()

    def _init_kernel(self):
        torch.manual_seed(self.kernel_seed)
        kernel_size = tuple(itertools.chain((self.out_channels,), self.kernel_size))
        self._kernel = torch.randn(kernel_size)

    def _unfold(self,x):
        B,c_in,h_in,w_in = x.shape
        assert self.in_channels == c_in
        h_out = math.floor(((h_in + 2*self.padding[0] - self.dilation[0]*(self.kernel_size[0]-1) - 1)/self.stride[0])+1)
        w_out = math.floor(((w_in + 2*self.padding[1] - self.dilation[1]*(self.kernel_size[1]-1) - 1)/self.stride[1])+1) 
        
        # unfold to create patches
        x_hat = torch.nn.functional.unfold(x, self.kernel_size, dilation=self.dilation, stride=self.stride) #Bx(CxKxK)x(H_out,W_out)

        # reshape
        x_hat = rearrange(x_hat, 'b (c k l) (h w) -> b c h w k l', c=self.in_channels,k=self.kernel_size[0],l=self.kernel_size[1],h=h_out,w=w_out)
        return x_hat
    
    def _conv(self,x):
        x_conv_k = torch.einsum('bcghkl, fkl -> bfgh',x,self._kernel)
        return x_conv_k

    def forward(self,x):
        x = self._unfold(x)
        x = self._conv(x)
        return x

In [149]:
B,c_in,h_in,w_in = map(int,x.shape)
c_out = 10 # number of kernels
ein_cov2d = EinConv2d(in_channels=c_in, out_channels=c_out, kernel_size=(3,3))
z = ein_cov2d(x)
print(z.shape)
z

torch.Size([2, 10, 2, 2])


tensor([[[[ -2.9415,   5.3964],
          [  5.4321,  -5.3896]],

         [[  8.2950,   2.9994],
          [ -0.5231,  -5.0381]],

         [[  3.9444,   2.6381],
          [ -2.4011,   3.7015]],

         [[ -0.3334,  -1.8386],
          [  2.7771,  -0.6299]],

         [[  1.7169,  -4.0328],
          [ -1.4772,  -5.0409]],

         [[  0.9792,   2.0948],
          [ -5.6420,  -2.1269]],

         [[  6.5511,  -1.9491],
          [  3.8565,  -5.2588]],

         [[ -0.3668,  -0.9743],
          [ -4.1852,  11.8087]],

         [[  2.3531,   0.2098],
          [  8.0488,   0.7861]],

         [[  1.2799,  -0.3592],
          [-10.9903,  10.7230]]],


        [[[  3.4981,  -2.4474],
          [  6.4705,   1.8358]],

         [[ -1.3067,  -4.3596],
          [  2.4444,  -2.0905]],

         [[ -2.2434,   1.9631],
          [ -3.9482,  -0.9244]],

         [[  1.9027,   0.8784],
          [ -0.6463,   0.9799]],

         [[ -0.8404,  -1.3682],
          [  2.2192,   3.7414]],

        

In [138]:
.shape

torch.Size([10, 3, 3])

## Validation?

In [152]:
y0000 = (x[:,:,:3,:3]*ein_cov2d._kernel[0]).sum((-3, -2,-1))
y0001 = (x[:,:,:3,1:4]*ein_cov2d._kernel[0]).sum((-3, -2,-1))
y0010 = (x[:,:,1:4,:3]*ein_cov2d._kernel[0]).sum((-3, -2,-1))
y0011 = (x[:,:,1:5,1:4]*ein_cov2d._kernel[0]).sum((-3, -2,-1))


y00 = (y0000,y0001,y0010,y0011)
for _y in y00:
    print(_y)

tensor([-2.9415,  3.4981])
tensor([ 5.3964, -2.4474])
tensor([5.4321, 6.4705])
tensor([-5.3896,  1.8358])
